# [센서 실습 5] 근접(장애물) 센서 – FC-51 적외선 장애물 센서

적외선 반사형 근접 센서 **FC-51**의 디지털 출력을 읽어 물체를 감지하고, **디바운싱**으로 잡음을 거른 뒤 지나간 물체 수를 셉니다. (컨베이어 물품 계수기)

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
FC-51은 **적외선 LED**가 빛을 쏘고, 앞에 물체가 있으면 반사된 빛을 **포토트랜지스터**가 받습니다.
- 받은 빛의 세기를 비교기(LM393)가 기준값과 비교 → **디지털 0/1**로 출력
- **물체 감지 시 LOW(0)**, 없으면 HIGH(1) – 이렇게 거꾸로 동작하는 것을 **액티브 로우(Active Low)** 라고 함
- 감지 거리(2~30cm)는 모듈의 가변저항으로 조절
- 검은색 물체는 빛을 흡수해 잘 감지되지 않음

### 부품 사양
| 항목 | 내용 |
|---|---|
| 감지 거리 | 2 ~ 30 cm (가변저항 조절) |
| 출력 | 디지털 (감지 시 LOW) |
| 전원 | 3.3 ~ 5V |
| 감지 각도 | 약 35° |
| 응답 | 수 ms |

## 2. 라즈베리파이 연결 방법
**준비물**: FC-51 모듈, 점퍼선 3개

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| FC-51 VCC | 3.3V (1번 핀) | 3.3V로 공급해야 출력도 3.3V |
| FC-51 GND | GND (6번 핀) |  |
| FC-51 OUT | GPIO17 (11번 핀) | 감지 시 0 |



**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
출력 핀의 전압 레벨(0/1)을 '감지 여부'로 바꿉니다. 액티브 로우이므로 **0이 감지**입니다.

In [ ]:
def is_detected(pin_level):
    """FC-51: 물체를 감지하면 LOW(0)를 출력 (액티브 로우)"""
    return int(pin_level == 0)

print("핀 0 →", is_detected(0), "/ 핀 1 →", is_detected(1))

### 4-2. 실제 센서 클래스
`gpiozero`의 `DigitalInputDevice`로 핀 레벨을 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealFC51:
    def __init__(self, pin=17):
        from gpiozero import DigitalInputDevice
        self.dev = DigitalInputDevice(pin, pull_up=None, active_state=True)   # 모듈이 직접 0/1을 출력

    def read(self):
        level = int(self.dev.value)
        return {"level": level, "detected": is_detected(level)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 컨베이어 벨트 위를 물품이 지나감 | 물품 계수기 |
| 물품 간격 | 1.5 ~ 4초 무작위 | 사람이 올려놓는 속도 |
| 감지 시간 | 0.3 ~ 0.8초 | 물품 길이 ÷ 벨트 속도 |
| 잡음 | 2% 확률로 0.1초짜리 오감지 | 햇빛·반사광에 의한 순간 오동작 |

출력은 실제 센서처럼 **핀 레벨(감지 시 0)** 로 만들어 변환식을 통과시킵니다.

In [ ]:
class VirtualFC51:
    def __init__(self, dt=0.1, seed=5):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)
        self.next_item, self.item_end = 1.0, -1.0

    def read(self):
        if self.t >= self.next_item:                               # 새 물품 도착
            self.item_end = self.t + self.rng.uniform(0.3, 0.8)
            self.next_item = self.t + self.rng.uniform(1.5, 4.0)
        present = self.t < self.item_end
        if not present and self.rng.random() < 0.02:               # 순간 오감지
            present = True
        level = 0 if present else 1                                # 감지 시 LOW
        self.t += self.dt
        return {"level": level, "detected": is_detected(level)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualFC51(dt=0.1)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
0.1초 간격으로 60초 동안 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 600               # 측정 횟수
INTERVAL = 0.1               # 측정 간격(초)

sensor = RealFC51() if USE_REAL_SENSOR else VirtualFC51(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. **디바운싱(debouncing)**: 연속 2회(0.2초) 이상 감지되어야 진짜 물체로 인정 → 순간 오감지 제거
2. 0→1로 바뀌는 순간(**상승 에지**)을 세면 지나간 물품 수가 됩니다.

In [ ]:
raw_edges = (df["detected"].diff() == 1).sum()
df["stable"] = (df["detected"].rolling(2).sum() == 2).astype(int)   # 2회 연속 감지
df["count"] = (df["stable"].diff() == 1).cumsum()

print(f"디바운싱 전 감지 횟수: {raw_edges}회")
print(f"디바운싱 후 물품 수 : {df['count'].iloc[-1]}개")
print(f"분당 처리량: {df['count'].iloc[-1] / (N_SAMPLES * INTERVAL / 60):.1f}개/분")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
ax[0].step(df["time_s"], df["level"], "k-", where="post"); ax[0].set_ylabel("핀 레벨")
ax[0].set_title("FC-51 근접 센서 – 물품 계수")
ax[1].step(df["time_s"], df["stable"], "k-", where="post")
ax[1].fill_between(df["time_s"], df["stable"], step="post", color="lightgray"); ax[1].set_ylabel("디바운싱 후")
ax[2].step(df["time_s"], df["count"], "k-", where="post"); ax[2].set_ylabel("누적 개수")
ax[2].set_xlabel("시간 (초)")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "근접센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "FC-51 적외선 장애물 센서",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/proximity", payload)
# client.disconnect()

## 직접 해 보기
- 디바운싱 기준을 3회 연속으로 바꾸면 결과가 어떻게 달라지나요?
- 오감지 확률을 10%로 올려 디바운싱의 효과를 비교해 보세요.
- 물품이 10개가 될 때마다 '상자 포장' 메시지를 출력해 보세요.